<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_6_3_%D0%92%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 6.3. Введение в LangChain

## Введение: от самодельного RAG к промышленному фреймворку

Поздравляю! Мы прошли огромный путь. В Лекции 6.1 мы создали игрушечного RAG-агента с нуля — вручную отправляли HTTP-запросы к Ollama, писали свой семантический поиск через `cosine_similarity` и маршрутизировали по ключевым словам. В Лекции 6.2 мы превратили его в полноценную систему: загрузка документов, умный чанкинг, векторная база Chroma, интеллектуальная маршрутизация через LLM, память и логирование. Всё это мы сделали **на чистом Python, без фреймворков**, чтобы понять, как всё работает «под капотом».

Теперь настало время **сделать шаг вперёд**. Мы освоим **LangChain** — мощный фреймворк для разработки приложений на основе больших языковых моделей. Он не заменит наше понимание, а **возьмёт на себя рутину**: управление промптами, цепочками вызовов, памятью, интеграцию с векторными базами и многое другое. LangChain позволяет сократить объём кода в 3–5 раз, делая его более читаемым, гибким и легко расширяемым.

**Что мы изучим в этой лекции:**
- **Компоненты LangChain** — LLM, промпты, цепочки, ретриверы, память.
- **LCEL (LangChain Expression Language)** — новый синтаксис для построения пайплайнов.
- **Вызов LLM через ChatOllama** — вместо `requests.post` одна строка кода.
- **Промпты и парсеры** — структурированное управление шаблонами и форматированием ответов.
- **Цепочки** — объединение промпта и LLM в единый конвейер.
- **Ретриверы и векторные хранилища** — интеграция с Chroma через LangChain.
- **Память** — управление историей диалога через встроенные классы.
- **Сборка полноценного RAG-агента** на LangChain.

Мы перепишем ту же систему, что строили в Лекции 6.2, но теперь с использованием абстракций LangChain. Вы увидите, как сокращается код и возрастает его выразительность.

---

## Тема 1. Что такое LangChain и зачем он нужен

LangChain — это фреймворк с открытым исходным кодом, созданный для упрощения разработки приложений на основе LLM. Он предоставляет стандартизированные интерфейсы для работы с моделями, данными и логикой приложений.

### 1.1. Ключевые компоненты LangChain

| Компонент | Назначение | В нашей системе из Лекции 6.2 |
|-----------|------------|-------------------------------|
| **LLM** | Интерфейс для языковых моделей (Ollama, OpenAI, Anthropic) | Мы делали `requests.post` вручную |
| **Промпт** | Управление шаблонами и форматированием | Мы вручную собирали строки с f-строками |
| **Цепочки (Chains)** | Объединение нескольких шагов в один пайплайн | Мы писали отдельные функции и вызывали их последовательно |
| **Ретриверы** | Поиск релевантных документов (интерфейс для векторных БД) | Мы сами писали `search_chunks` |
| **Память (Memory)** | Хранение истории диалога | Мы реализовали `ConversationMemory` вручную |
| **Парсеры (Parsers)** | Извлечение структурированных данных из ответов LLM | Мы парсили JSON вручную с `json.loads` |

Каждый компонент предоставляет **стандартизированный интерфейс**. Это значит, что мы можем легко заменить одну модель на другую, одну векторную БД на другую, не переписывая остальной код.

### 1.2. LCEL — LangChain Expression Language

LCEL — это синтаксический сахар для построения цепочек. Вместо того чтобы писать:

```python
prompt = PromptTemplate(...)
chain = prompt | llm | parser
result = chain.invoke({"input": "вопрос"})
```

Мы используем оператор `|` для объединения компонентов. Это напоминает конвейеры в Unix (`|`) и делает код **декларативным** — мы описываем **что** делаем, а не **как**.

**Пример простейшей цепочки:**

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen2.5:3b")

prompt = ChatPromptTemplate.from_template("Ответь на вопрос: {question}")

chain = prompt | llm

response = chain.invoke({"question": "Что такое LangChain?"})
print(response.content)
```

Этот код делает ровно то же, что и 10 строк с `requests.post`, но лаконичнее и стандартизированнее.

### 1.3. Установка пакетов

Для работы с LangChain в нашем проекте установим необходимые пакеты:

```bash
pip install langchain langchain-community langchain-chroma langchain-ollama
```

**Что мы установили:**

| Пакет | Назначение |
|-------|------------|
| `langchain` | Ядро фреймворка (базовые интерфейсы) |
| `langchain-community` | Интеграции с различными сервисами (Chroma, HuggingFace, и др.) |
| `langchain-chroma` | Специализированный пакет для работы с Chroma через LangChain |
| `langchain-ollama` | Интеграция с Ollama через стандартный интерфейс LangChain |

Теперь проверим, что всё установилось корректно:

```bash
python -c "import langchain; import langchain_ollama; print('✅ LangChain готов к работе')"
```

### 1.4. Обзорная схема: как всё будет связано

Вот как будет выглядеть наша RAG-система на LangChain:

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                          ПОЛЬЗОВАТЕЛЬ                                      │
│                                │                                             │
│                           Вопрос: "Что такое ProjectFlow?"                  │
└────────────────────────────────┬────────────────────────────────────────────┘
                                 ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                         CHAIN (ЦЕПОЧКА)                                   │
│                                                                             │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  1. ПРОМПТ С ИСТОРИЕЙ                                            │    │
│   │  - Шаблон: "ИСТОРИЯ: {history}\nВОПРОС: {question}\nОТВЕТ:"     │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  2. РЕТРИВЕР (VectorStoreRetriever)                              │    │
│   │  - Ищет в Chroma топ‑3 чанка по вопросу                         │    │
│   │  - Возвращает: [документ1, документ2, документ3]                │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  3. ПРОМПТ С КОНТЕКСТОМ (RAG)                                    │    │
│   │  - Шаблон: "КОНТЕКСТ: {context}\nВОПРОС: {question}"             │    │
│   │  - Подставляем найденные чанки                                    │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  4. LLM (ChatOllama)                                             │    │
│   │  - Отправляет запрос в qwen2.5:3b                                │    │
│   └───────────────────────────────────────────────────────────────────┘    │
│                                   │                                         │
│                                   ▼                                         │
│   ┌───────────────────────────────────────────────────────────────────┐    │
│   │  5. ПАРСЕР (StrOutputParser)                                     │    │
│   │  - Извлекает response.content в виде строки                      │    │
│   └───────────────────────────────────────────────────────────────────┘    │
└─────────────────────────────────────────────────────────────────────────────┘
                                 │
                                 ▼
┌─────────────────────────────────────────────────────────────────────────────┐
│                          ПОЛЬЗОВАТЕЛЬ                                      │
│                                                                             │
│                    Ответ: "ProjectFlow — это облачная платформа..."        │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Вся цепочка на LangChain будет выглядеть примерно так:**

```python
chain = (
    {"context": retriever, "question": lambda x: x["question"]}
    | rag_prompt
    | llm
    | StrOutputParser()
)
```

И это **вся логика RAG**. Остальное — загрузка документов и настройка векторной базы, но это тоже делается в несколько строк.

Теперь, когда мы понимаем концепцию, давайте начнём с самого простого — вызова LLM через LangChain.

---




## Тема 2. Вызов LLM через ChatOllama (продолжение — подробно о параметрах)

В предыдущем разделе мы кратко перечислили параметры `ChatOllama`. Теперь давайте разберём каждый из них **максимально подробно** — как они работают, какие значения принимать и как влияют на результат.

### 2.6. Полный список параметров ChatOllama

```python
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:3b",           # Имя модели
    temperature=0.0,               # 0 = детерминированно, 1 = креативно
    num_predict=1024,              # Максимум токенов в ответе
    top_k=40,                      # Ограничение на выбор топ-K токенов
    top_p=0.9,                     # Ядерная выборка (nucleus sampling)
    repeat_penalty=1.1,            # Штраф за повторения
    stop=["\n", "Вопрос:"],        # Стоп-слова
    timeout=60,                    # Таймаут в секундах
    num_ctx=4096,                  # Размер контекстного окна (токены)
    seed=None,                     # Детерминированность (фиксированный seed)
    tfs_z=1.0,                     # Tail-free sampling (Z-значение)
    mirostat_mode=0,               # Режим Mirostat (0=выкл, 1=вкл, 2=экспериментальный)
    mirostat_tau=5.0,              # Температура Mirostat
    mirostat_eta=0.1,              # Скорость обучения Mirostat
)
```

Теперь разберём каждый параметр.

---

### 2.6.1. `model` — выбор модели

**Что это:** строка с именем модели, загруженной в Ollama.

**Примеры:**
- `"qwen2.5:3b"`
- `"llama3.1:8b"`
- `"mistral:7b"`

**Как использовать:** перед инициализацией убедитесь, что модель скачана:

```bash
ollama pull qwen2.5:3b
```

Если модель не найдена, вы получите ошибку соединения.

**Почему важно:** разные модели имеют разный размер, качество и скорость. Для экспериментов подойдёт `qwen2.5:3b` (2 ГБ), для серьёзных задач — `llama3.1:8b` (4.7 ГБ).

---

### 2.6.2. `temperature` — контроль случайности

**Что это:** число от 0 до 2 (чаще 0–1). Влияет на креативность модели.

| Значение | Эффект |
|----------|--------|
| `0.0` | **Детерминированно** — каждый раз один и тот же ответ (при прочих равных). Идеально для точных фактов, кода, математики. |
| `0.3–0.5` | **Сбалансированно** — небольшая вариативность. Хорошо для чатов, когда нужна естественность, но без фантазий. |
| `0.7–0.9` | **Креативно** — модель генерирует разные варианты. Для творческих задач, генерации идей, сторителлинга. |
| `1.0+` | **Очень креативно** — может уходить в сторону, иногда галлюцинировать. |

**Как это работает технически:** LLM на каждом шаге предсказывает вероятности для всех токенов. `temperature` масштабирует эти вероятности — чем выше значение, тем «площе» распределение, и модель с большей вероятностью выбирает не самый очевидный токен. При `temperature=0` выбирается токен с максимальной вероятностью (жадный поиск).

**Пример:**

```python
# Детерминированный ответ (всегда один)
llm_det = ChatOllama(model="qwen2.5:3b", temperature=0.0)
for _ in range(3):
    print(llm_det.invoke("Скажи число 2+2").content)  # всегда "4"

# Креативный ответ (может меняться)
llm_creative = ChatOllama(model="qwen2.5:3b", temperature=0.9)
for _ in range(3):
    print(llm_creative.invoke("Придумай имя для кота").content)
    # Вывод: "Барсик", "Мурзик", "Снежок" — разные варианты
```

---

### 2.6.3. `num_predict` — максимальная длина ответа

**Что это:** максимальное количество токенов в ответе (не включая промпт).

| Значение | Применение |
|----------|------------|
| `128` | Короткие ответы (да/нет, одно слово, краткая справка) |
| `512` | Стандартный ответ средней длины (параграф) |
| `1024` | Развёрнутый ответ (несколько абзацев) |
| `2048+` | Длинные тексты (статьи, обзоры, код) |

**Важно:** не путать с размером контекстного окна (`num_ctx`). `num_predict` ограничивает только **генерируемый ответ**, а `num_ctx` — объём входных данных, которые модель может «увидеть» (включая промпт и историю).

**Пример:** если промпт занимает 3000 токенов, а `num_ctx=4096`, то модель может сгенерировать ещё 1096 токенов. Но если `num_predict=512`, то ответ будет обрезан после 512 токенов, даже если модель могла бы продолжить.

---

### 2.6.4. `top_k` — фильтрация по количеству

**Что это:** число (обычно 10–100). Модель рассматривает только `top_k` наиболее вероятных токенов на каждом шаге.

| Значение | Эффект |
|----------|--------|
| `1` | Жадный поиск (всегда самый вероятный токен) — аналог `temperature=0` |
| `10–20` | Умеренное ограничение — модель выбирает из небольшого пула лучших токенов |
| `40–100` | Широкий выбор — больше разнообразия |
| `0` (или очень большое) | Отключает фильтрацию — рассматриваются все токены (медленнее) |

**Как работает:** на каждом шаге модель вычисляет вероятности для всех токенов (десятки тысяч). `top_k` оставляет только K самых вероятных, остальные обнуляет. Это ускоряет генерацию и убирает маловероятный «мусор».

**Пример:** при `top_k=10` модель никогда не выберет редкое слово, даже если оно контекстуально подходит, если оно не входит в топ‑10. При `top_k=40` такой токен имеет шанс.

---

### 2.6.5. `top_p` — ядерная выборка (nucleus sampling)

**Что это:** число от 0 до 1 (обычно 0.8–0.95). Модель выбирает минимальный набор токенов, чья суммарная вероятность превышает `top_p`.

| Значение | Эффект |
|----------|--------|
| `0.1` | Очень строго — только самые вероятные токены (почти детерминированно) |
| `0.5` | Умеренно — половина вероятностной массы |
| `0.9` | Широкий выбор — 90% массы (обычно это десятки–сотни токенов) |
| `1.0` | Без ограничений (эквивалент `top_p=1`) |

**Разница между `top_k` и `top_p`:**
- `top_k` — фильтрует по **количеству** токенов (фиксированное число).
- `top_p` — фильтрует по **кумулятивной вероятности** (динамическое число).

Обычно используют **либо `top_k`, либо `top_p`**, но можно комбинировать: сначала `top_k` сужает список, затем `top_p` отсекает по вероятности.

**Пример:** если вероятности токенов [0.5, 0.3, 0.15, 0.05], то при `top_p=0.9` будут выбраны первые три (сумма 0.95), четвёртый отброшен.

---

### 2.6.6. `repeat_penalty` — штраф за повторения

**Что это:** число от 1.0 до 2.0 (обычно 1.05–1.2). Уменьшает вероятность повторения уже использованных токенов.

| Значение | Эффект |
|----------|--------|
| `1.0` | Штраф выключен — модель может повторяться |
| `1.05–1.1` | Лёгкий штраф — немного уменьшает повторения |
| `1.2–1.5` | Сильный штраф — активно избегает повторений |
| `2.0+` | Почти запрещает повторения (может сломать логику) |

**Как работает:** модель умножает вероятности токенов, которые уже встречались в контексте, на `1/repeat_penalty`. Чем выше штраф, тем меньше шанс выбрать повторяющийся токен.

**Пример:**
- `repeat_penalty=1.0`: модель может написать «Я думаю, что я думаю...»
- `repeat_penalty=1.2`: такая фраза маловероятна.

---

### 2.6.7. `stop` — стоп-слова

**Что это:** список строк, при обнаружении которых генерация останавливается.

**Пример:** `stop=["\n", "Вопрос:"]` — ответ будет оборван, как только встретится символ новой строки или слово «Вопрос:».

**Применение:**
- Чтобы ограничить ответ одним предложением (`stop=["."]`).
- Чтобы запретить модели задавать встречные вопросы (`stop=["?"]`).
- Чтобы разделять несколько ответов в одном запросе.

---

### 2.6.8. `timeout` — таймаут запроса

**Что это:** максимальное время ожидания ответа от Ollama в секундах.

**Значение:** если модель не отвечает за `timeout` секунд, выбрасывается исключение `TimeoutError`.

**Рекомендация:** для коротких промптов — 30 сек, для длинных генераций (например, статей) — 120+ сек.

---

### 2.6.9. `num_ctx` — размер контекстного окна

**Что это:** количество токенов, которое модель может «видеть» (вход + выход вместе). Это **не** то же самое, что `num_predict`.

| Значение | Применение |
|----------|------------|
| `2048` | Минимальное (многие старые модели) |
| `4096` | Стандарт для многих моделей (qwen2.5:3b поддерживает 8192) |
| `8192` | Расширенное окно (для длинных документов) |
| `32768` | Для моделей с большим контекстом (Llama 3.1) |

**Важно:** если суммарный объём (промпт + история + ответ) превышает `num_ctx`, модель обрежет начало промпта. Поэтому для RAG-систем с большими документами нужно увеличивать `num_ctx`.

---

### 2.6.10. `seed` — детерминизм

**Что это:** целое число (или `None`). Задаёт начальное состояние генератора случайных чисел.

| Значение | Эффект |
|----------|--------|
| `None` | Случайный seed — ответы будут различаться при каждом запуске (даже при `temperature=0`) |
| `42` | Фиксированный seed — ответы будут идентичны при одинаковых параметрах |

**Применение:** для воспроизводимости экспериментов, отладки, тестирования.

---

### 2.6.11. `tfs_z` — Tail-free sampling

**Что это:** число (обычно 1.0). Альтернативный метод фильтрации токенов.

- `1.0` — выключено.
- `<1.0` — более жёсткая фильтрация (отбрасывает «хвост» распределения).

Редко используется, оставьте по умолчанию.

---

### 2.6.12. `mirostat_mode`, `mirostat_tau`, `mirostat_eta` — алгоритм Mirostat

**Что это:** экспериментальный алгоритм управления случайностью, предложенный исследователями Microsoft. Позволяет автоматически подстраивать температуру во время генерации, чтобы достичь заданной «энтропии» (разнообразия).

- `mirostat_mode=0` — выключено.
- `mirostat_mode=1` — включён (рекомендуется для творческих задач).
- `mirostat_tau` — целевая энтропия (обычно 5.0). Чем выше, тем разнообразнее текст.
- `mirostat_eta` — скорость адаптации (0.1 — стандарт).

**Применение:** для генерации художественных текстов, где нужно балансировать между разнообразием и связностью.

---

### 2.7. Практические рекомендации по выбору параметров

| Сценарий | Параметры |
|----------|-----------|
| **Точные факты (RAG, QA)** | `temperature=0.0`, `top_k=1` или `top_p=0.1`, `repeat_penalty=1.0` |
| **Код (генерация, объяснение)** | `temperature=0.2`, `top_p=0.5`, `repeat_penalty=1.05` |
| **Чат (естественный диалог)** | `temperature=0.7`, `top_p=0.9`, `repeat_penalty=1.1` |
| **Креативное письмо** | `temperature=0.9`, `top_p=0.95`, `mirostat_mode=1`, `mirostat_tau=5.0` |
| **Сжатие/пересказ** | `temperature=0.3`, `num_predict=256`, `stop=["."]` |

---

### 2.8. Полный код с демонстрацией параметров

Создадим файл `test_llm_params.py`:

```python
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage

# Набор конфигураций
configs = [
    {"name": "Точный (детерминированный)", "temp": 0.0, "top_k": 1},
    {"name": "Сбалансированный", "temp": 0.5, "top_p": 0.8},
    {"name": "Креативный", "temp": 0.9, "top_p": 0.95},
]

question = "Придумай название для стартапа, который делает ИИ-помощника для программистов."

for cfg in configs:
    llm = ChatOllama(
        model="qwen2.5:3b",
        temperature=cfg["temp"],
        top_k=cfg.get("top_k", 40),
        top_p=cfg.get("top_p", 0.9),
        repeat_penalty=1.1,
        num_predict=128,
        seed=42  # Фиксируем seed для воспроизводимости (кроме температуры)
    )
    
    response = llm.invoke([HumanMessage(content=question)])
    print(f"\n{cfg['name']} (temp={cfg['temp']}):")
    print(response.content)
    print("-" * 60)
```

**Вывод:**
```
Точный (детерминированный) (temp=0.0):
CodeMate

------------------------------------------------------------
Сбалансированный (temp=0.5):
DevPal AI

------------------------------------------------------------
Креативный (temp=0.9):
СodeCraft AI — твой интеллектуальный партнёр в мире программирования!

------------------------------------------------------------
```

Мы видим, как меняется ответ в зависимости от параметров.

---

## Краткий итог по параметрам

| Параметр | Влияние на генерацию |
|----------|----------------------|
| `temperature` | Креативность (чем выше, тем разнообразнее) |
| `num_predict` | Длина ответа |
| `top_k`/`top_p` | Фильтрация вероятностей (чем строже, тем детерминированнее) |
| `repeat_penalty` | Борьба с повторениями |
| `stop` | Принудительная остановка генерации |
| `num_ctx` | Объём контекста (вход + выход) |
| `seed` | Воспроизводимость |

Теперь, когда мы вооружены знанием каждого параметра, можем тонко настраивать модель под любую задачу. В следующей теме мы перейдём к **промптам LangChain** — ещё одному мощному инструменту, который делает наш код компактным и выразительным.



## Тема 3. Промпты и парсеры: от ручного форматирования к структурированным шаблонам

В Лекциях 6.1 и 6.2 мы формировали промпты вручную, используя f-строки и многострочные литералы:

```python
prompt = f"""
Ты — строгий помощник. Отвечай на основе контекста.

Контекст: {context}
Вопрос: {question}
Ответ:
"""
```

Этот подход, хотя и работает, обладает рядом фундаментальных недостатков, которые становятся критическими по мере роста сложности системы:

1. **Смешение логики и представления.** Шаблон промпта размазан по коду, его трудно редактировать, тестировать и поддерживать. Изменение структуры промпта требует правки кода, что нарушает принцип разделения ответственности.

2. **Отсутствие структурной типизации.** Мы не различаем системные инструкции, историю диалога и текущий вопрос — всё смешивается в единую строку. Это особенно проблематично для чат-моделей, которые ожидают структурированный диалог.

3. **Ручной парсинг ответов.** Для маршрутизации мы вручную парсили JSON с помощью `json.loads()` и обрабатывали ошибки через `try/except`. Это порождает шаблонный код, который сложно поддерживать.

4. **Нет валидации.** Мы не проверяли, что модель вернула корректный JSON с нужными полями. Любая ошибка в ответе модели приводила либо к падению, либо к неявному fallback-значению.

5. **Сложность переиспользования.** Один и тот же промпт невозможно легко применить к разным сценариям без копирования кода.

**LangChain предлагает элегантное решение всех этих проблем через три ключевых механизма:**

1. **Шаблоны промптов** (`ChatPromptTemplate`) — отделяют структуру промпта от логики приложения, позволяют задавать разные роли (система, пользователь, ассистент) и подставлять переменные.

2. **Парсеры** (`StrOutputParser`, `PydanticOutputParser`) — автоматически извлекают и валидируют ответы модели, превращая их в типизированные объекты Python.

3. **Интеграция с LCEL** — позволяет собирать промпты, парсеры и модели в единые цепочки, делая код декларативным и легко тестируемым.

В этой теме мы детально разберём каждый из этих механизмов и увидим, как они трансформируют наш подход к разработке RAG-систем.

---

### 3.1. ChatPromptTemplate: создание структурированных шаблонов

`ChatPromptTemplate` — это класс для создания промптов из нескольких сообщений с возможностью подстановки переменных. В отличие от обычных строк, он поддерживает разделение ролей и структурную типизацию.

#### Базовое использование

Самый простой способ — использование фабричного метода `from_template()`:

```python
from langchain_core.prompts import ChatPromptTemplate

# Шаблон с одной переменной
prompt = ChatPromptTemplate.from_template("Ответь на вопрос: {question}")

# Шаблон с несколькими переменными
prompt = ChatPromptTemplate.from_template(
    "Контекст: {context}\nВопрос: {question}\nОтвет:"
)

# Подстановка значений
formatted = prompt.format(question="Что такое LangChain?")
print(formatted)
```

**Вывод:**
```
Ответь на вопрос: Что такое LangChain?
```

#### Создание шаблонов с несколькими сообщениями

Наиболее мощный вариант — создание шаблона из нескольких сообщений с явным указанием ролей. Это соответствует формату, который ожидают современные чат-модели:

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — эксперт по {topic}. Отвечай кратко и чётко."),
    ("human", "Объясни: {concept}"),
])

# Подстановка
formatted = prompt.format(topic="Python", concept="декоратор")
print(formatted)
```

**Вывод:**
```
System: Ты — эксперта по Python. Отвечай кратко и чётко.
Human: Объясни: декоратор
```

#### Преимущества перед f-строками

| Аспект | f-строки (ручные) | ChatPromptTemplate |
|--------|-------------------|-------------------|
| **Разделение ролей** | Отсутствует | Чёткое (system, human, assistant) |
| **Переиспользование** | Копирование кода | Один шаблон — много вызовов |
| **Поддержка** | Трудно изменять | Легко редактировать |
| **Тестируемость** | Низкая | Высокая (шаблоны изолированы) |
| **Интеграция с LCEL** | Нет | Полная поддержка |

#### Форматирование с явными сообщениями

Иногда удобнее работать с объектами сообщений напрямую. LangChain предоставляет классы `SystemMessage`, `HumanMessage`, `AIMessage`:

```python
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Создаём список сообщений вручную
messages = [
    SystemMessage(content="Ты — помощник по программированию."),
    HumanMessage(content="Что такое класс в Python?"),
]

# Передаём в LLM
response = llm.invoke(messages)
print(response.content)
```

Этот подход особенно полезен, когда мы динамически строим историю диалога, добавляя сообщения ассистента между сообщениями пользователя.

---

### 3.2. Использование SystemMessage, HumanMessage, AIMessage для явного указания ролей

В LangChain каждое сообщение имеет явную роль, что критически важно для корректной работы чат-моделей. Рассмотрим три основных типа:

**1. SystemMessage (Системное сообщение)**

Задаёт контекст, правила и ограничения для модели. Обычно находится в начале диалога и определяет поведение модели на протяжении всей сессии.

```python
from langchain_core.messages import SystemMessage

system_msg = SystemMessage(content="Ты — эксперт по Python. Отвечай кратко и только по теме.")
```

**2. HumanMessage (Сообщение пользователя)**

Представляет запрос или вопрос пользователя. Модель должна ответить на это сообщение.

```python
from langchain_core.messages import HumanMessage

human_msg = HumanMessage(content="Что такое декоратор в Python?")
```

**3. AIMessage (Сообщение ассистента)**

Представляет ответ модели. Используется в истории диалога для сохранения контекста.

```python
from langchain_core.messages import AIMessage

ai_msg = AIMessage(content="Декоратор — это функция, которая принимает другую функцию...")
```

**Комбинирование с ChatPromptTemplate:**

```python
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="Ты — эксперт по {topic}."),
    HumanMessage(content="{question}"),
])

# Подстановка
formatted = prompt.format_messages(topic="Python", question="Что такое декоратор?")
# formatted — список сообщений, готовых для передачи в LLM
```

**Почему это важно:** Многие модели (особенно из семейства Llama, Qwen) чувствительны к структуре диалога. Явное разделение ролей помогает модели лучше понимать, что от неё требуется.

---

### 3.3. PydanticOutputParser: автоматический парсинг структурированных ответов

Одна из самых мощных возможностей LangChain — автоматический парсинг ответов в заданную структуру с помощью Pydantic. Это полностью устраняет необходимость вручную парсить JSON и обрабатывать ошибки.

#### Что такое Pydantic?

Pydantic — это библиотека для валидации данных через Python-классы с аннотациями типов. Она автоматически проверяет, что данные соответствуют заданной структуре, и преобразует их в объекты Python.

#### Создание модели данных

Определим структуру для маршрутизации запросов:

```python
from pydantic import BaseModel, Field
from typing import Literal

class RouterOutput(BaseModel):
    """Структура ответа маршрутизатора."""
    action: Literal["search", "answer"] = Field(
        description="Действие: search — искать в документах, answer — ответить из знаний"
    )
    confidence: float = Field(
        description="Уверенность в решении (0.0 — 1.0)",
        ge=0.0,  # минимальное значение
        le=1.0   # максимальное значение
    )
```

**Разбор полей:**
- `action` — строка, которая может принимать только два значения: `"search"` или `"answer"` (используем `Literal` для ограничения).
- `confidence` — число с плавающей точкой от 0 до 1 (используем `ge` и `le` для ограничения диапазона).
- `Field(description=...)` — добавляет описание, которое автоматически попадает в инструкцию для модели.

#### Создание парсера

```python
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=RouterOutput)
```

#### Получение инструкций для модели

Парсер автоматически генерирует инструкции по форматированию ответа:

```python
format_instructions = parser.get_format_instructions()
print(format_instructions)
```

**Вывод (упрощённый):**
```
The output should be formatted as a JSON instance that conforms to the JSON schema below.

Here is the output schema:
{"properties": {"action": {"description": "Action: search — искать в документах, answer — ответить из знаний", "enum": ["search", "answer"], "title": "Action", "type": "string"}, "confidence": {"description": "Уверенность в решении (0.0 — 1.0)", "maximum": 1.0, "minimum": 0.0, "title": "Confidence", "type": "number"}}, "required": ["action", "confidence"]}
```

Эту инструкцию мы вставляем в промпт, чтобы модель точно знала, как форматировать ответ.

---

### 3.4. Полный цикл: маршрутизация через PydanticParser

Теперь соберём все компоненты в единую систему. Создадим файл `router_with_parser.py`:

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

# ============================================================================
# 1. Определяем структуру ответа с помощью Pydantic
# ============================================================================

class RouterOutput(BaseModel):
    """Структура ответа маршрутизатора."""
    action: Literal["search", "answer"] = Field(
        description="Действие: search — искать в документах, answer — ответить из знаний"
    )
    confidence: float = Field(
        description="Уверенность в решении (0.0 — 1.0)",
        ge=0.0,
        le=1.0
    )

# ============================================================================
# 2. Создаём парсер
# ============================================================================

parser = PydanticOutputParser(pydantic_object=RouterOutput)

# ============================================================================
# 3. Создаём промпт с инструкциями от парсера
# ============================================================================

prompt = ChatPromptTemplate.from_messages([
    ("system", """
Ты — интеллектуальный маршрутизатор запросов в RAG-системе.

Твоя задача — определить, нужно ли искать информацию в базе знаний, или можно ответить на основе собственных знаний.

База знаний содержит документы о продукте ProjectFlow, финансовые отчёты и историю компании.

Правила принятия решения:
1. Если вопрос требует фактов, цифр или специфической информации из документов → action = "search"
2. Если вопрос общий (математика, философия, погода, юмор) → action = "answer"
3. Если вопрос является уточнением к предыдущему → учитывай контекст (даже если он не сохранён явно)

Верни ответ в строгом формате JSON, следуя инструкциям ниже.

{format_instructions}
"""),
    ("human", "Вопрос: {question}"),
])

# ============================================================================
# 4. Инициализируем LLM с оптимальными параметрами
# ============================================================================

llm = ChatOllama(
    model="qwen2.5:3b",
    temperature=0.0,      # Детерминированно для маршрутизации
    num_predict=128,      # Короткий ответ — достаточно
    top_p=0.1,            # Ограничиваем выбор для стабильности
)

# ============================================================================
# 5. Собираем цепочку с помощью LCEL
# ============================================================================

chain = prompt | llm | parser

# ============================================================================
# 6. Тестируем на разных вопросах
# ============================================================================

test_questions = [
    "Что такое ProjectFlow?",
    "Какая выручка компании в первом квартале?",
    "Сколько будет 2+2?",
    "Как работает гравитация?",
]

print("=" * 60)
print("ТЕСТИРОВАНИЕ МАРШРУТИЗАТОРА С PYDANTICPARSER")
print("=" * 60)

for question in test_questions:
    print(f"\n{'=' * 60}")
    print(f"Вопрос: {question}")
    print('-' * 60)
    
    try:
        # Вызываем цепочку — результат сразу будет объектом RouterOutput
        result = chain.invoke({
            "question": question,
            "format_instructions": parser.get_format_instructions()
        })
        
        # result — уже объект RouterOutput с валидированными полями!
        print(f"✅ Решение: {result.action}")
        print(f"   Уверенность: {result.confidence:.2f}")
        print(f"   Тип результата: {type(result)}")
        print(f"   Поля: action={result.action}, confidence={result.confidence}")
        
    except Exception as e:
        print(f"❌ Ошибка парсинга: {e}")
        print("   (Это защита от невалидного ответа модели)")
```

#### Реальный вывод при запуске

Вот что мы видим при выполнении скрипта в терминале:

```
(.venv) PS D:\Science\AI_Agent_Demo> python router_with_parser.py

============================================================
ТЕСТИРОВАНИЕ МАРШРУТИЗАТОРА С PYDANTICPARSER
============================================================

============================================================
Вопрос: Что такое ProjectFlow?
------------------------------------------------------------
✅ Решение: search
   Уверенность: 0.90
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=search, confidence=0.9

============================================================
Вопрос: Какая выручка компании в первом квартале?
------------------------------------------------------------
✅ Решение: search
   Уверенность: 0.90
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=search, confidence=0.9

============================================================
Вопрос: Сколько будет 2+2?
------------------------------------------------------------
✅ Решение: answer
   Уверенность: 1.00
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=answer, confidence=1.0

============================================================
Вопрос: Как работает гравитация?
------------------------------------------------------------
✅ Решение: answer
   Уверенность: 1.00
   Тип результата: <class '__main__.RouterOutput'>
   Поля: action=answer, confidence=1.0
(.venv) PS D:\Science\AI_Agent_Demo>
```

**Анализ результатов:**

- Вопросы о ProjectFlow и выручке корректно направляются на поиск (`search`) с уверенностью 0.90.
- Общие вопросы (математика, физика) направляются на ответ из знаний (`answer`) с максимальной уверенностью 1.00.
- Все результаты автоматически валидированы и представлены как объекты Python.

---

### 3.5. Сравнение с ручным подходом из Лекции 6.2

Чтобы оценить преимущества LangChain, давайте сравним код маршрутизатора из Лекции 6.2 и новой версии с PydanticParser.

#### Лекция 6.2 (ручной парсинг JSON)

```python
import json
import requests

def router_manual(question):
    prompt = f"""
    Ты — маршрутизатор. Верни JSON: {{"action": "search" или "answer", "confidence": 0.0-1.0}}
    Вопрос: {question}
    JSON:
    """
    
    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": "qwen2.5:3b",
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.0}
            },
            timeout=30
        )
        raw_text = response.json().get("response", "")
    except Exception as e:
        print(f"Ошибка запроса: {e}")
        return "search", 0.5  # Fallback
    
    try:
        # Ищем JSON в ответе
        start = raw_text.find('{')
        end = raw_text.rfind('}') + 1
        
        if start == -1 or end == 0:
            raise ValueError("JSON не найден")
        
        json_str = raw_text[start:end]
        data = json.loads(json_str)
        
        action = data.get("action", "search")
        confidence = data.get("confidence", 0.5)
        
        # Валидация вручную
        if action not in ["search", "answer"]:
            action = "search"
        if not (0.0 <= confidence <= 1.0):
            confidence = 0.5
            
        return action, confidence
        
    except (json.JSONDecodeError, ValueError, KeyError) as e:
        print(f"Ошибка парсинга: {e}")
        return "search", 0.5  # Fallback
```

#### Лекция 6.3 (LangChain с PydanticParser)

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

# Определяем структуру
class RouterOutput(BaseModel):
    action: Literal["search", "answer"] = Field(description="Действие")
    confidence: float = Field(description="Уверенность", ge=0.0, le=1.0)

# Создаём парсер
parser = PydanticOutputParser(pydantic_object=RouterOutput)

# Создаём промпт
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты — маршрутизатор. {format_instructions}"),
    ("human", "Вопрос: {question}"),
])

# Инициализируем LLM
llm = ChatOllama(model="qwen2.5:3b", temperature=0.0)

# Собираем цепочку
chain = prompt | llm | parser

# Вызываем
result = chain.invoke({
    "question": "Что такое ProjectFlow?",
    "format_instructions": parser.get_format_instructions()
})

# result — готовый объект RouterOutput!
print(result.action)      # "search"
print(result.confidence)  # 0.9
```

#### Сравнительная таблица

| Аспект | Лекция 6.2 (ручной) | Лекция 6.3 (LangChain) |
|--------|---------------------|----------------------|
| **Объём кода** | 40+ строк | 15 строк |
| **Обработка ошибок** | Ручная (`try/except` для запроса, парсинга, валидации) | Автоматическая (встроена в парсер) |
| **Валидация** | Ручная (проверка полей) | Автоматическая (Pydantic) |
| **Типизация** | Нет (словари) | Есть (объекты Pydantic) |
| **Документация** | Нет | Автоматическая через `Field(description=...)` |
| **Гибкость** | Низкая (правка логики во многих местах) | Высокая (правка одной модели) |
| **Интеграция с цепочками** | Нет | Полная (LCEL) |
| **Повторное использование** | Копипаст | Один класс — много сценариев |
| **Надёжность** | Средняя (зависит от качества ручного парсинга) | Высокая (строгая валидация) |

---

### 3.6. Дополнительные возможности PydanticOutputParser

#### Вложенные структуры

Pydantic позволяет создавать сложные вложенные структуры:

```python
from pydantic import BaseModel
from typing import List

class Document(BaseModel):
    title: str
    content: str
    relevance_score: float

class SearchResult(BaseModel):
    query: str
    documents: List[Document]
    total_count: int

parser = PydanticOutputParser(pydantic_object=SearchResult)
```

#### Кастомные валидаторы

Можно добавить собственные проверки:

```python
from pydantic import BaseModel, validator

class RouterOutput(BaseModel):
    action: str
    confidence: float
    
    @validator('confidence')
    def validate_confidence(cls, v):
        if not (0.0 <= v <= 1.0):
            raise ValueError(f'confidence must be between 0 and 1, got {v}')
        return v
```

#### Обработка ошибок парсинга

LangChain предоставляет механизм для обработки ошибок парсинга:

```python
from langchain_core.output_parsers import OutputFixingParser

# Создаём парсер, который может исправлять ошибки
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)

# Используем в цепочке
chain = prompt | llm | fixing_parser
```

---

## Краткий итог Тема 3

- Мы освоили **ChatPromptTemplate** — инструмент для создания структурированных промптов с разделением ролей (system, human, assistant). Это позволяет отделить логику от представления и делает код более поддерживаемым.

- Научились использовать **PydanticOutputParser** — автоматический парсинг ответов в Python-объекты с валидацией. Это полностью исключает ручной парсинг JSON и обработку ошибок.

- Создали полноценный маршрутизатор, который возвращает структурированный ответ без единой строки `json.loads()` или `try/except` для парсинга.

- Увидели, как код стал **чище, короче и надёжнее** по сравнению с ручным подходом из Лекции 6.2. Объём кода сократился в 2–3 раза, а надёжность выросла благодаря автоматической валидации.

- Изучили дополнительные возможности Pydantic: вложенные структуры, кастомные валидаторы и обработку ошибок парсинга.

Теперь у нас есть все необходимые кирпичики для сборки RAG-цепочки:
- **LLM** — `ChatOllama` для вызова модели.
- **Промпты** — `ChatPromptTemplate` для структурированных шаблонов.
- **Парсеры** — `PydanticOutputParser` для автоматической валидации и типизации.

В следующей теме мы объединим их с помощью LCEL и добавим ретривер для поиска в документах, создав полноценную RAG-систему на LangChain. Оставайтесь с нами!